# Übung 1.2: Klassifikation und Clustering auf einem kleinen räumlichen Datensatz

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/01_machine_learning/german_version/exercise_1_2_classification_clustering_de.ipynb)

Dieses Notebook gehört zur zweiten Machine-Learning-Einheit. Wir verwenden einen kleinen synthetischen georäumlichen Datensatz und vergleichen mehrere Methoden aus der Vorlesung:

- überwachte Klassifikation: logistische Regression, SVM, Decision Tree, Random Forest und Gradient Boosting
- unüberwachtes Clustering: K-Means, hierarchisches Clustering, Gaussian Mixture Models und DBSCAN
- Modellbewertung mit Accuracy, Macro-F1, Confusion Matrix, Silhouette Score und Adjusted Rand Index
- nebeneinander dargestellte Visualisierung mehrerer Clustering-Methoden auf denselben Daten

Der Datensatz ist synthetisch, aber wie eine kleine Stadtanalyse aufgebaut: Jeder Punkt ist ein Standort mit lokalen Koordinaten und einfachen Gebietseigenschaften.


## 0. Setup

Führen Sie dies zuerst in Colab aus. Das Notebook erzeugt die Daten lokal; es ist kein externer Datendownload nötig.


In [ ]:
%pip -q install scikit-learn pandas matplotlib seaborn

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    adjusted_rand_score,
    classification_report,
    confusion_matrix,
    f1_score,
    silhouette_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)


## 1. Einen kleinen räumlichen Datensatz erzeugen

Die Punkte repräsentieren Standorte in einem kleinen Stadtgebiet. Das Koordinatensystem ist lokal und in Kilometern von einem fiktiven Stadtzentrum gemessen.

Die vier Klassen sind absichtlich nicht perfekt getrennt. Dadurch wird der Vergleich linearer und nichtlinearer Klassifikatoren interessanter.


In [ ]:
def make_urban_sample(random_state=42):
    rng = np.random.default_rng(random_state)
    rows = []

    specs = [
        {
            "zone_type": "Residential",
            "n": 42,
            "mean": (-2.2, -1.1),
            "cov": [[0.35, 0.10], [0.10, 0.22]],
            "road": 3.8,
            "green": 0.32,
            "transit": 0.42,
        },
        {
            "zone_type": "Business center",
            "n": 42,
            "mean": (0.0, 0.0),
            "cov": [[0.28, -0.04], [-0.04, 0.24]],
            "road": 7.4,
            "green": 0.10,
            "transit": 0.86,
        },
        {
            "zone_type": "Green / recreation",
            "n": 38,
            "mean": (1.9, 1.55),
            "cov": [[0.42, 0.06], [0.06, 0.28]],
            "road": 1.9,
            "green": 0.78,
            "transit": 0.28,
        },
    ]

    for spec in specs:
        points = rng.multivariate_normal(spec["mean"], spec["cov"], size=spec["n"])
        for x, y in points:
            distance = np.sqrt(x**2 + y**2)
            rows.append(
                {
                    "x_km": x,
                    "y_km": y,
                    "distance_to_center_km": distance,
                    "road_density": np.clip(rng.normal(spec["road"], 0.45), 0, None),
                    "green_share": np.clip(rng.normal(spec["green"], 0.07), 0, 1),
                    "transit_access": np.clip(rng.normal(spec["transit"], 0.08), 0, 1),
                    "zone_type": spec["zone_type"],
                }
            )

    # A corridor-shaped class creates a less spherical shape. This is useful for testing SVMs and DBSCAN.
    t = rng.uniform(-2.7, 2.6, size=42)
    x = t + rng.normal(0, 0.24, size=t.size)
    y = 0.52 * t + 0.55 + rng.normal(0, 0.22, size=t.size)
    for xi, yi in zip(x, y):
        distance = np.sqrt(xi**2 + yi**2)
        rows.append(
            {
                "x_km": xi,
                "y_km": yi,
                "distance_to_center_km": distance,
                "road_density": np.clip(rng.normal(6.0, 0.55), 0, None),
                "green_share": np.clip(rng.normal(0.18, 0.06), 0, 1),
                "transit_access": np.clip(rng.normal(0.92, 0.05), 0, 1),
                "zone_type": "Mobility corridor",
            }
        )

    return pd.DataFrame(rows).sample(frac=1, random_state=random_state).reset_index(drop=True)

urban_df = make_urban_sample(RANDOM_STATE)
display(urban_df.head())
print(urban_df["zone_type"].value_counts())


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=urban_df,
    x="x_km",
    y="y_km",
    hue="zone_type",
    palette="Set2",
    s=55,
    edgecolor="white",
    linewidth=0.5,
    ax=ax,
)
ax.set_title("Small synthetic urban dataset")
ax.set_xlabel("Local x coordinate (km)")
ax.set_ylabel("Local y coordinate (km)")
ax.legend(title="Known class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()


### Aufgabe 1: Klassen inspizieren

Betrachten Sie den Scatterplot und beantworten Sie kurz:

- Welche Klassen lassen sich nur anhand der Lage gut trennen?
- Welche Klassen überlappen?
- Welche Klassenform passt am wenigsten zu einer einfachen centroid-basierten Methode?

<details>
<summary>Antwort / Tipp</summary>

`Residential` und `Green / recreation` sind meist leichter nach Lage zu trennen. `Business center` und `Mobility corridor` überlappen stärker, weil beide nahe am Zentrum liegen.

`Mobility corridor` passt am wenigsten zu einer einfachen centroid-basierten Methode. Die Klasse ist langgestreckt, während K-Means eher kompakte Cluster um ein Zentrum bevorzugt.
</details>


## 2. Train/Test-Split für Klassifikation

Wir starten nur mit den zwei räumlichen Koordinaten. Dadurch bleiben Entscheidungsgrenzen gut visualisierbar.

Später können Sie Attributspalten hinzufügen und prüfen, ob sich die Metriken verbessern.


In [ ]:
spatial_features = ["x_km", "y_km"]
attribute_features = ["distance_to_center_km", "road_density", "green_share", "transit_access"]

X = urban_df[spatial_features]
y = urban_df["zone_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.28,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)}")
print(f"Test rows:     {len(X_test)}")


## 3. Mehrere Klassifikationsmethoden vergleichen

Die ausgewählten Methoden entsprechen den Themen der Vorlesung:

- logistische Regression als einfache lineare Baseline
- SVM mit RBF-Kernel für nichtlineare Grenzen
- Decision Tree als interpretierbares regelbasiertes Modell
- Random Forest als Bagging-Ensemble von Decision Trees
- Gradient Boosting als sequenzielles Tree-Ensemble


### Accuracy und Macro-F1

`accuracy` ist der Anteil korrekter Vorhersagen.

`macro_f1` ist der durchschnittliche F1-Score über alle Klassen. Jede Klasse erhält dasselbe Gewicht. Ein Modell kann daher nicht nur deshalb gut aussehen, weil es die größte Klasse gut vorhersagt. Das ist nützlich, wenn Klassengrößen ungleich sind oder kleine Klassen wichtig sind.


In [ ]:
classifiers = {
    "Logistic regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    ),
    "SVM (RBF kernel)": make_pipeline(
        StandardScaler(),
        SVC(kernel="rbf", C=3.0, gamma="scale", random_state=RANDOM_STATE),
    ),
    "Decision tree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=250,
        max_depth=6,
        random_state=RANDOM_STATE,
    ),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

classification_results = []
trained_classifiers = {}

for name, model in classifiers.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    trained_classifiers[name] = model
    classification_results.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro"),
        }
    )

results_df = pd.DataFrame(classification_results).sort_values("macro_f1", ascending=False)
display(results_df)
print(f"Best model by macro F1: {results_df.iloc[0]['model']}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results_long = results_df.melt(id_vars="model", value_vars=["accuracy", "macro_f1"])
sns.barplot(data=results_long, x="value", y="model", hue="variable", ax=ax)
ax.set_xlim(0, 1)
ax.set_xlabel("Score")
ax.set_ylabel("")
ax.set_title("Classification performance on the test set")
plt.show()


## 4. Klassifikationsgrenzen visualisieren

Ein hoher Score ist nützlich, aber die Entscheidungsfläche erklärt, was der Klassifikator gelernt hat. Vergleichen Sie besonders lineare Baseline, SVM, Tree und Ensemble-Methoden.


In [ ]:
def plot_decision_boundaries(models, X_train, y_train, X_test, y_test):
    class_names = sorted(y_train.unique())
    class_to_id = {label: i for i, label in enumerate(class_names)}
    class_palette = dict(zip(class_names, sns.color_palette("Set2", n_colors=len(class_names))))
    contour_colors = [class_palette[label] for label in class_names]

    x_min, x_max = urban_df["x_km"].min() - 0.5, urban_df["x_km"].max() + 0.5
    y_min, y_max = urban_df["y_km"].min() - 0.5, urban_df["y_km"].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 280), np.linspace(y_min, y_max, 240))
    grid = pd.DataFrame({"x_km": xx.ravel(), "y_km": yy.ravel()})

    fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
    axes = axes.ravel()

    for ax, (name, model) in zip(axes, models.items()):
        pred = model.predict(grid)
        zz = np.array([class_to_id[p] for p in pred]).reshape(xx.shape)
        ax.contourf(
            xx,
            yy,
            zz,
            levels=np.arange(len(class_names) + 1) - 0.5,
            colors=contour_colors,
            alpha=0.28,
        )
        sns.scatterplot(
            x=X_train["x_km"],
            y=X_train["y_km"],
            hue=y_train,
            hue_order=class_names,
            palette=class_palette,
            s=28,
            linewidth=0,
            alpha=0.75,
            legend=False,
            ax=ax,
        )
        sns.scatterplot(
            x=X_test["x_km"],
            y=X_test["y_km"],
            hue=y_test,
            hue_order=class_names,
            palette=class_palette,
            s=72,
            edgecolor="black",
            linewidth=0.8,
            legend=False,
            ax=ax,
        )
        ax.set_title(name)
        ax.set_xlabel("x_km")
        ax.set_ylabel("y_km")

    axes[-1].axis("off")
    handles = [
        plt.Line2D([0], [0], marker="o", color="w", label=label, markerfacecolor=class_palette[label], markersize=8)
        for label in class_names
    ]
    axes[-1].legend(handles=handles, title="Class", loc="center")
    fig.suptitle("Decision boundaries: region colors and point colors use the same class mapping", y=1.02)
    plt.tight_layout()
    plt.show()

plot_decision_boundaries(trained_classifiers, X_train, y_train, X_test, y_test)


### Aufgabe 2: Klassifikationsgrenzen vergleichen

Wählen Sie zwei Klassifikatoren und vergleichen Sie:

- Welcher hat die glattere Grenze?
- Welcher overfittet diesen kleinen Datensatz eher?
- Ändern Sie bei der SVM `C` von `3.0` zu `0.5` und dann zu `20.0`. Was ändert sich?
- Ändern Sie beim Decision Tree `max_depth`. Was ändert sich?

<details>
<summary>Antwort / Tipp</summary>

Die logistische Regression hat meist die glatteste und einfachste Grenze, weil sie linear ist. Eine SVM mit RBF-Kernel kann die Grenze krümmen. Decision Trees erzeugen blockartige, scharfe Grenzen; tiefere Trees overfitten leichter.

Kleineres `C` macht die SVM meist glatter und toleranter gegenüber Fehlern. Größeres `C` versucht stärker, Trainingsdaten korrekt zu klassifizieren, und kann komplexer werden.
</details>


## 5. Den besten Klassifikator inspizieren

Wir wählen ein Modell mit Macro-F1. Macro-F1 gibt jeder Klasse dasselbe Gewicht, was bei unterschiedlichen Klassengrößen hilfreich ist.

Eine Confusion Matrix zeigt anschließend, welche Klassen miteinander verwechselt werden.


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_classifiers[best_model_name]
y_best = best_model.predict(X_test)

print(f"Best model by macro F1: {best_model_name}\n")
print(classification_report(y_test, y_best))

fig, ax = plt.subplots(figsize=(6.5, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_best,
    xticks_rotation=35,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title(f"Confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()


### Optionale Feature-Engineering-Aufgabe

Wiederholen Sie die Klassifikation mit allen Features:

```python
all_features = spatial_features + attribute_features
X = urban_df[all_features]
```

Verbessern sich die Scores? Welches Modell profitiert am meisten von den zusätzlichen Variablen?

<details>
<summary>Antwort / Tipp</summary>

Die Attributfeatures sollten meist helfen, weil sie Informationen enthalten, die in Koordinaten allein nicht sichtbar sind: Straßendichte, Grünanteil und ÖPNV-Zugang. Tree-basierte Modelle und Random Forests profitieren oft von solchen zusätzlichen Attributen.
</details>


## 6. Dieselben Daten für Clustering vorbereiten

Clustering ist unüberwacht: Der Algorithmus erhält das Label `zone_type` nicht.

Wir behalten die bekannten Labels trotzdem zurück, damit wir nachträglich prüfen können, ob die gefundenen Gruppen den erzeugten Klassen ähneln. In echter explorativer Arbeit gibt es diese Ground Truth oft nicht.


In [ ]:
cluster_features = ["x_km", "y_km", "road_density", "green_share", "transit_access"]
X_cluster = urban_df[cluster_features]
X_cluster_scaled = StandardScaler().fit_transform(X_cluster)

true_label_ids = LabelEncoder().fit_transform(urban_df["zone_type"])
print("Clustering features:", cluster_features)


## 7. Mehrere Clustering-Methoden ausführen

Die Methoden repräsentieren Clustering-Familien aus der Vorlesung:

- K-Means: centroid-basiert, benötigt `k`
- Agglomeratives Clustering: hierarchisch, hier ebenfalls auf `k=4` geschnitten
- GMM / EM: verteilungsbasiert, ordnet Punkte Gauß-Komponenten zu
- DBSCAN: dichtebasiert, kann Punkte mit Label `-1` als Noise markieren

Die Tabelle nutzt zwei Clustering-Scores:

- `silhouette`: höher ist besser. Misst, ob Punkte näher an ihrem eigenen Cluster liegen als an anderen Clustern. Verwendet keine echten Labels.
- `adjusted_rand_index` (ARI): höher ist besser; `1.0` bedeutet perfekte Übereinstimmung mit den bekannten Labels. ARI verwendet `zone_type` nur nach dem Clustering zur Evaluation.


In [ ]:
clusterers = {
    "K-Means (k=4)": KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE),
    "Agglomerative (k=4)": AgglomerativeClustering(n_clusters=4, linkage="ward"),
    "GMM / EM (k=4)": GaussianMixture(n_components=4, covariance_type="full", random_state=RANDOM_STATE),
    "DBSCAN": DBSCAN(eps=0.72, min_samples=6),
}

cluster_labels = {}
cluster_scores = []

for name, method in clusterers.items():
    labels = method.fit_predict(X_cluster_scaled)
    cluster_labels[name] = labels

    n_clusters = len(set(labels) - {-1})
    has_valid_silhouette = len(set(labels)) > 1 and n_clusters > 0
    silhouette = silhouette_score(X_cluster_scaled, labels) if has_valid_silhouette else np.nan
    ari = adjusted_rand_score(true_label_ids, labels)

    cluster_scores.append(
        {
            "method": name,
            "clusters_found": n_clusters,
            "noise_points": int(np.sum(labels == -1)),
            "silhouette": silhouette,
            "adjusted_rand_index": ari,
        }
    )

cluster_scores_df = pd.DataFrame(cluster_scores).sort_values("adjusted_rand_index", ascending=False)
display(cluster_scores_df)


## 8. Clustering-Visualisierung im Vergleich

Alle Methoden verwenden dieselben Eingabepunkte. Nur der Clustering-Algorithmus ändert sich.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10), sharex=True, sharey=True)
axes = axes.ravel()

for ax, (name, labels) in zip(axes, cluster_labels.items()):
    plot_df = urban_df.copy()
    plot_df["cluster"] = labels.astype(str)
    plot_df["is_noise"] = labels == -1

    non_noise = plot_df[~plot_df["is_noise"]]
    noise = plot_df[plot_df["is_noise"]]

    sns.scatterplot(
        data=non_noise,
        x="x_km",
        y="y_km",
        hue="cluster",
        palette="tab10",
        s=50,
        edgecolor="white",
        linewidth=0.45,
        legend=False,
        ax=ax,
    )
    if not noise.empty:
        ax.scatter(
            noise["x_km"],
            noise["y_km"],
            c="black",
            marker="x",
            s=55,
            label="noise",
        )

    score_row = cluster_scores_df[cluster_scores_df["method"] == name].iloc[0]
    ax.set_title(
        f"{name}\nclusters={score_row['clusters_found']}, "
        f"noise={score_row['noise_points']}, ARI={score_row['adjusted_rand_index']:.2f}"
    )
    ax.set_xlabel("Local x coordinate (km)")
    ax.set_ylabel("Local y coordinate (km)")

fig.suptitle("Clustering methods on the same small spatial dataset", y=1.02)
plt.tight_layout()
plt.show()


### Aufgabe 3: Clustering-Ergebnisse vergleichen

Vergleichen Sie die vier Clustering-Plots:

- Welche Methode erfasst die langgestreckte Corridor-Klasse am besten?
- Welche Methoden zwingen jeden Punkt in ein Cluster?
- Welche Methode kann Noise-Punkte markieren?
- Ändern Sie `eps` in DBSCAN auf `0.55`, `0.85` und `1.10`. Was passiert mit Clusterzahl und Noise?
- Ändern Sie `k` / `n_components` von `4` auf `3` oder `5`. Welches Ergebnis wirkt plausibler?

<details>
<summary>Antwort / Tipp</summary>

K-Means, agglomeratives Clustering und GMM ordnen jeden Punkt einem Cluster zu. DBSCAN kann Punkte als Noise mit Label `-1` markieren.

Die langgestreckte Corridor-Klasse ist schwierig für centroid-basierte Methoden. DBSCAN kann Teile davon besser erfassen, ist aber empfindlich gegenüber `eps` und `min_samples`.
</details>


## 9. Verbindung zur Vorlesung

Nutzen Sie diese Checkliste für eine kurze Zusammenfassung:

- Überwachtes Lernen nutzt gelabelte Beispiele und wird auf einem zurückgehaltenen Testset bewertet.
- SVM konzentriert sich auf Trennfläche und Margin; der Kernel steuert die Flexibilität der Grenze.
- Decision Trees erzeugen explizite Regeln; Ensembles wie Random Forest und Gradient Boosting reduzieren oft Schwächen einzelner Trees.
- Unüberwachtes Clustering sucht Struktur ohne Klassenlabels.
- K-Means und GMM benötigen eine gewählte Anzahl von Gruppen; DBSCAN nutzt Dichteparameter.
- Die beste Clustering-Methode hängt stark von Clusterform, Dichte, Skalierung und Analyseziel ab.
